# Serial Dictatorship with Coalitions

This notebook tests the serial dictatorship mechanism extended to handle coalitions in a student-project matching setting. It uses the repository's `src` modules and validates outcomes for coalition-based assignment scenarios.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.instance import Instance, Student, Project
from core.coalitions import CoalitionInstance, Coalition
from mechanisms.serial_dictatorship_for_coalition import coalition_serial_dictatorship
from mechanisms.serial_dictatorship import serial_dictatorship

print('Imports successful')


## Define Preference Profile and Coalition Utilities

Create helper functions to build instances, present preferences, and define coalition structures.

In [ ]:
def build_instance(students, projects, preferences):
    return Instance(
        students=tuple(Student(s) for s in students),
        projects=tuple(Project(p_id, capacity=capacity) for p_id, capacity in projects),
        preferences={s: pref[:] for s, pref in preferences.items()}
    )

def print_matching(matching):
    print('Assignments:')
    for student_id, project_id in sorted(matching.assignments.items()):
        print(f'  Student {student_id} -> Project {project_id}')

def coalition_member_ids(coalitions):
    return [Coalition(tuple(sorted(group))) for group in coalitions]

print('Utility functions defined')


## Implement Serial Dictatorship with Coalitions

Use the repository's coalition-based serial dictatorship implementation and compare it with standard serial dictatorship.

In [ ]:
def run_coalition_sd(instance, coalition_groups, order=None):
    coalition_instance = CoalitionInstance(instance, coalition_groups)
    return coalition_serial_dictatorship(coalition_instance, order=order)

def run_standard_sd(instance, order=None):
    return serial_dictatorship(instance, order=order)

print('Coalition serial dictatorship wrapper ready')


## Create Test Cases for Coalition Scenarios

Define example scenarios with coalitions and preferences to validate assignment behavior.

In [ ]:
# Scenario 1: One coalition of size 2 plus one single student
students = [1, 2, 3]
projects = [(10, 2), (20, 1)]
preferences = {
    1: [10, 20],
    2: [10, 20],
    3: [20, 10],
}
coalitions = [(1, 2)]
instance_1 = build_instance(students, projects, preferences)

# Scenario 2: Two coalitions and no singles
students = [1, 2, 3, 4]
projects = [(10, 2), (20, 2)]
preferences = {
    1: [10, 20],
    2: [10, 20],
    3: [20, 10],
    4: [20, 10],
}
coalitions_2 = [(1, 2), (3, 4)]
instance_2 = build_instance(students, projects, preferences)

# Scenario 3: Mixed coalition and single with stronger preferences
students = [1, 2, 3, 4]
projects = [(10, 2), (20, 2)]
preferences = {
    1: [20, 10],
    2: [20, 10],
    3: [10, 20],
    4: [10, 20],
}
coalitions_3 = [(1, 2)]
instance_3 = build_instance(students, projects, preferences)

print('Test cases created')


## Evaluate Outcomes Across Profiles and Coalitions

Run each scenario with coalition serial dictatorship and standard serial dictatorship, then compare the results.

In [ ]:
def evaluate_scenario(instance, coalition_groups, scenario_name):
    print(f'--- {scenario_name} ---')
    coalition_matching = run_coalition_sd(instance, coalition_groups)
    standard_matching = run_standard_sd(instance)

    print('Coalition serial dictatorship result:')
    print_matching(coalition_matching)
    print('Standard serial dictatorship result:')
    print_matching(standard_matching)

    if coalition_matching.assignments == standard_matching.assignments:
        print('Result: Assignments match the standard serial dictatorship')
    else:
        print('Result: Assignments differ from standard serial dictatorship')
    print()

evaluate_scenario(instance_1, coalitions, 'Scenario 1: Coalition of size 2 + single')
evaluate_scenario(instance_2, coalitions_2, 'Scenario 2: Two coalitions')
evaluate_scenario(instance_3, coalitions_3, 'Scenario 3: Coalition with different preferences')

print('Evaluation complete')
